# Comparison between Bayesian EVT and Possibilistic EVT

In [28]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import time
import pandas as pd

import pymc as pm
import arviz as az

from tqdm import tqdm

np.random.seed(42)

## 1. Bayesian inference

In [29]:
def simulate_gpd(n, xi, sigma, threshold=0):
    var = stats.genpareto.rvs(c=xi, scale=sigma, size=n) + threshold
    return var



In [30]:
def fit_bayesian_gpd(data, draws=2000, tune=1000):
    with pm.Model() as model:

        #prior
        xi = pm.Normal("xi", mu=0, sigma=0.5)
        sigma = pm.HalfNormal("sigma", sigma=2.0)

        #likelihood
        pm.Potential("likelihood", pm.logp(pm.GeneralizedPareto.dist(mu=0, sigma=sigma, xi=xi), data))

        trace = pm.sample(draws=draws, tune=tune, chains=2, progressbar=False)
    
    return trace

In [31]:
def posterior_predictive_extreme(trace, size=1000):
    xi_samples = trace.posterior["xi"].values.flatten()
    sigma_samples = trace.posterior["sigma"].values.flatten()

    samples = []
    for xi, sigma in zip(xi_samples, sigma_samples):
        samples.append(stats.genpareto.rvs(c=xi, scale=sigma))
    
    return np.array(samples)

In [32]:
def true_quantile(p, xi, sigma):
    return stats.genpareto.ppf(p, c=xi, scale=sigma)

def estimated_quantiles(samples, p):
    return np.quantiles(samples, p)

In [33]:
def return_level(trace, T):
    xi = trace.posterior["xi"].values.flatten()
    sigma = trace.posterior["sigma"].values.flatten()

    return sigma / xi * ((T**xi) - 1)

In [34]:
def calibration(true_value, samples, alpha=0.95):
    lower = np.quantile(samples, (1-alpha)/2)
    upper = np.quantile(samples, 1 - (1-alpha)/2)
    return lower <= true_value <= upper

In [35]:
def run_experiment(n, xi_true, sigma_true, p=0.99):
    data = simulate_gpd(n, xi_true, sigma_true)

    # fit
    start = time.time()
    trace = fit_bayesian_gpd(data)
    runtime = time.time() - start

    pred_samples = posterior_predictive_extreme(trace)

    q_true = true_quantile(p, xi_true, sigma_true)
    q_est = estimated_quantiles(pred_samples, p)

    # metrics
    error = np.abs(q_est - q_true)
    calibration = calibration(q_true, pred_samples)

    return {"n":n, "error": error, "calibration": calibration, "runtime": runtime, "q_true": q_true, "q_est": q_est}

results = []

for n in [20, 50, 100, 500]:
    for xi_true in [0.1, 0.3, 0.5]:
        for _ in tqdm(range(10)):
            res = run_experiment(n, xi_true, sigma_true=1.0)
            results.append(res)

  0%|          | 0/10 [00:00<?, ?it/s]


AttributeError: module 'pymc' has no attribute 'GeneralizedPareto'

In [ ]:
df = pd.DataFrame(results)

plt.figure()
for xi in df["n"].unique():
    subset = df[df["n"] == xi]
    plt.scatter(subset["n"], subset["error"], alpha=0.5)

plt.xscale("log")
plt.xlable("Sample size")
plt.ylabel("Quantile error")
plt.title("Tail estimation error")
plt.show()

In [ ]:
calibration_rates = df.groupby("n")["calibration"].mean()

plt.figure()
plt.plot(calibration_rates.index, calibration_rates.values, marker='o')
plt.axhline(0.95, linestyle='--')
plt.xlabel("Sample size")
plt.ylabel("Empirical calibration")
plt.title("Calibration of Bayesian GPD")
plt.show()

In [ ]:
#posterior uncertainty visualization

az.plot_posterior(trace)
plt.show()

In [ ]:
interval_width = np.quantile(pred_samples, 0.975) - np.quantile(pred_samples, 0.025)